# FAC_NB_Resumen_Bitacora — cierre de corrida: qué se cargó y con qué marca

Paso final del pipeline (y útil a mano durante la demo). Muestra la **bitácora de una corrida**, el **catálogo con la marca de agua** y cuántas filas cargó cada corrida en `fac_bronze_transacciones`.

## Controles (hacen fallar el cuaderno para que el pipeline no "termine bien" con una carga a medias)

- Con `run_id` (así lo llama el pipeline), la corrida debe tener **una fila de bitácora por cada tabla activa esperada** (las tablas activas del catálogo, filtradas por `filtro_tabla`): detecta una iteración que murió antes de registrar nada.
- Si hay **alguna fila en `ERROR`**, el cuaderno falla.
- Con `run_id`, si no hay ninguna tabla activa que coincida, también falla.

> Importante: este paso se ejecuta aunque el ForEach haya fallado (dependencia *Completed*). Si no fallara él mismo ante un error, el pipeline podría quedar en verde con una tabla rota.

`run_id` vacío = **modo consulta**: se resume la corrida más reciente de la bitácora y solo se revisan los errores (no se compara con el catálogo, porque se desconoce el filtro con el que se lanzó esa corrida). Con el `run_id` que pasa el pipeline se aplican todos los controles.

In [ ]:
# Celda de parámetros
run_id = ""            # vacío = última corrida registrada en la bitácora
filtro_tabla = ""      # mismo filtro que se pasó al pipeline
destino = "auto"       # auto | ninguno | nombre de esquema

## 1. Bitácora de la corrida y verificación de integridad

In [ ]:
import json
import re
import uuid
import datetime
from pyspark.sql import functions as F

PATRON_IDENT = re.compile(r"[A-Za-z][A-Za-z0-9_]{0,63}\Z")


def validar_ident(valor, campo):
    """Los nombres se interpolan en SQL: solo se aceptan identificadores simples."""
    if not isinstance(valor, str) or PATRON_IDENT.fullmatch(valor) is None:
        raise ValueError(
            f"{campo}: se esperaba un identificador (letra inicial; letras, números y _); se recibió {valor!r}"
        )
    return valor


def resolver_esquema(destino):
    """auto = 'fac' si el Lakehouse tiene esquemas habilitados, si no ninguno; 'ninguno' o un nombre explícito."""
    d = (destino or "auto").strip()
    if d.lower() == "ninguno":
        return ""
    if d.lower() == "auto":
        try:
            nombres = {str(r[0]).lower() for r in spark.sql("SHOW SCHEMAS").collect()}
        except Exception:
            nombres = set()
        return "fac" if "dbo" in nombres else ""
    return validar_ident(d, "destino")


def calificar(esquema, tabla):
    return f"{esquema}.{tabla}" if esquema else tabla

import notebookutils

run_id = (run_id or "").strip()
modo_consulta = not run_id   # sin run_id = consulta interactiva de la última corrida: no se compara con el catálogo
filtro_tabla = (filtro_tabla or "").strip()
if filtro_tabla:
    validar_ident(filtro_tabla, "filtro_tabla")

ESQUEMA = resolver_esquema(destino)
CAT = calificar(ESQUEMA, "fac_catalogo_ingesta")
BIT = calificar(ESQUEMA, "fac_bitacora_ingesta")
for t in (CAT, BIT):
    if not spark.catalog.tableExists(t):
        raise RuntimeError(f"No existe {t}. Ejecuta primero FAC_NB_Preparar_Control.")

bitacora = spark.table(BIT)
if not run_id:
    ultima = bitacora.orderBy(F.col("inicio").desc()).select("run_id").first()
    if ultima is None:
        raise RuntimeError("La bitácora está vacía: no hay corridas que resumir.")
    run_id = ultima[0]
print("run_id:", run_id)

filas_run = bitacora.where(F.col("run_id") == run_id).orderBy("inicio")
print("== Bitácora de la corrida ==")
display(filas_run)
filas = filas_run.collect()

activas = spark.table(CAT).where(F.col("activo") == 1)
if filtro_tabla:
    activas = activas.where(F.col("tabla_destino") == filtro_tabla)
esperadas = {r[0] for r in activas.select("tabla_destino").collect()}
procesadas = {r["tabla"] for r in filas}
errores = [r for r in filas if r["estado"] == "ERROR"]

problemas = []
if not modo_consulta:
    if not esperadas:
        problemas.append("no hay tablas activas que coincidan con el filtro")
    if esperadas - procesadas:
        problemas.append(f"tablas sin fila en la bitácora: {sorted(esperadas - procesadas)}")
if errores:
    problemas.append("tablas con ERROR: " + ", ".join(f"{r['tabla']} ({(r['mensaje'] or '')[:120]})" for r in errores))

## 2. Catálogo con la marca de agua y filas cargadas por corrida

Aquí se ve el efecto del incremental: `ultima_marca` avanza solo cuando hubo filas nuevas y cada corrida deja su huella en `_fac_run_id`.

In [ ]:
print("== Catálogo (marca de agua) ==")
display(spark.table(CAT).select("orden", "tabla_destino", "tipo_carga", "columna_marca", "ultima_marca", "activo").orderBy("orden"))

TX = calificar(ESQUEMA, "fac_bronze_transacciones")
tx_por_corrida, tx_total = {}, None
if spark.catalog.tableExists(TX):
    tx_por_corrida = {r[0]: int(r[1]) for r in spark.table(TX).groupBy("_fac_run_id").count().collect()}
    tx_total = sum(tx_por_corrida.values())
    print("== Filas de fac_bronze_transacciones por corrida de carga ==")
    display(spark.table(TX).groupBy("_fac_run_id").agg(F.count(F.lit(1)).alias("filas"), F.min("_fac_cargado_en").alias("cargado_en")).orderBy("cargado_en"))
    print("Total de filas en la tabla:", spark.table(TX).count())

## 3. Resultado y salida

Si hay problemas se falla (después de mostrarlo todo). Si no, `exit()` devuelve un JSON resumen. `exit()` va en su propia celda y fuera de `try/except`.

In [ ]:
resumen = {
    "run_id": run_id,
    "tablas_esperadas": sorted(esperadas),
    "tablas_procesadas": sorted(procesadas),
    "estados": {r["tabla"]: r["estado"] for r in filas},
    "filas_leidas": {r["tabla"]: int(r["filas_leidas"]) for r in filas},
    "filas_escritas": {r["tabla"]: int(r["filas_escritas"]) for r in filas},
    "marca_despues": {r["tabla"]: r["marca_despues"] for r in filas if r["marca_despues"]},
    "problemas": problemas,
    "catalogo_marcas": {r["tabla_destino"]: r["ultima_marca"] for r in spark.table(CAT).collect()},
    "transacciones_total": tx_total,
    "transacciones_por_corrida": tx_por_corrida,
}
print(json.dumps(resumen, ensure_ascii=False, indent=2))
if problemas:
    raise RuntimeError("Resumen de la corrida con problemas: " + " | ".join(problemas))
notebookutils.notebook.exit(json.dumps(resumen, ensure_ascii=False))